# Data Cleaning -- Test Dataset
**This notebook prepares Flatiron Health CSV files for patients with metastatic RCC in anticipation of training a gradient-boosted survival model to predict mortality from time of first-line treatment. Specifically, it processes and cleans the test cohort.**

## Import packages

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from flatiron_cleaner import DataProcessorRenal, merge_dataframes

## Clean CSV files 

In [2]:
test_ids = pd.read_csv('../outputs/test_patient_ids.csv')

In [3]:
test_ids = test_ids.PatientID.to_list()

In [4]:
index_date_df = pd.read_csv('../data/LineOfTherapy.csv')

In [5]:
index_date_df = (
    index_date_df
    .query('LineNumber == 1')
    [['PatientID', 'StartDate']]
)

In [6]:
index_date_df.shape

(8337, 2)

In [7]:
index_date_df = (
    index_date_df[index_date_df.PatientID.isin(test_ids)]
    [['PatientID', 'StartDate']]
)

In [8]:
index_date_df.shape

(1668, 2)

In [9]:
processor = DataProcessorRenal()

### Process Enhanced_MetastaticRCC.csv

In [10]:
enhanced_df = processor.process_enhanced(file_path = '../data/Enhanced_MetastaticRCC.csv',
                                         index_date_df = index_date_df,
                                         index_date_column = 'StartDate',
                                         drop_dates = False)

2026-03-16 09:59:49,169 - INFO - Successfully read Enhanced_MetastaticRCC.csv file with shape: (10766, 10) and unique PatientIDs: 10766
2026-03-16 09:59:49,173 - INFO - Successfully filtered Enhanced_MetastaticRCC.csv file with shape: (1668, 11) and unique PatientIDs: 1668
2026-03-16 09:59:49,181 - INFO - Successfully processed Enhanced_MetastaticCRC.csv file with final shape: (1668, 13) and unique PatientIDs: 1668


In [11]:
enhanced_df = enhanced_df.copy()

In [12]:
# remove patients with missing MetDiagnosisDate
enhanced_df = enhanced_df.query('MetDiagnosisDate.notna()')

In [13]:
enhanced_df['days_met_to_treatment'] = (enhanced_df['imported_StartDate'] - enhanced_df['MetDiagnosisDate']).dt.days
enhanced_df['days_met_to_treatment'] = np.where(enhanced_df['days_met_to_treatment'] < 0 , 0, enhanced_df['days_met_to_treatment'])

enhanced_df['days_diag_to_treatment'] = (enhanced_df['imported_StartDate'] - enhanced_df['DiagnosisDate']).dt.days
enhanced_df['days_diag_to_treatment'] = np.where(enhanced_df['days_diag_to_treatment'] < 0 , 0, enhanced_df['days_diag_to_treatment'])

In [14]:
enhanced_df['days_diagnosis_to_met'] = np.where(enhanced_df['days_diagnosis_to_met'] < 0 , 0, enhanced_df['days_diagnosis_to_met'])
enhanced_df['days_diagnosis_to_met'] = enhanced_df['days_diagnosis_to_met'].fillna(0)

In [15]:
enhanced_df['SmokingStatus'] = enhanced_df['SmokingStatus'].map({
    'History of smoking': 1,
    'No history of smoking': 0,
    'Unknown/Not documented': 1
})

In [16]:
enhanced_df['StageFourDetail'] = enhanced_df['StageFourDetail'].fillna('Unknown')

In [17]:
enhanced_df['NephrectomyType_mod'] = enhanced_df['NephrectomyType_mod'].map(
    lambda x: 'other/unknown' if x in ['Unknown', 'Other'] else x
)

enhanced_df['NephrectomyType_mod'] = enhanced_df['NephrectomyType_mod'].astype('category')

In [18]:
histology_mapping = {
    'Clear Cell': 'clear_cell',
    'RCC, NOS': 'nos',  
    'Papillary': 'papillary',
    'Chromophobe': 'chromophobe',
    'Collecting Duct': 'rare_aggressive',
    'Renal Medullary': 'rare_aggressive',  
    'Translocation': 'rare_other',
    'Other': 'rare_other',
    'Non-Clear Cell Carcinoma': 'rare_other'
}

enhanced_df['Histology_grouped'] = enhanced_df['Histology'].map(histology_mapping).astype('category')
enhanced_df = enhanced_df.drop(columns = ['Histology'])

In [19]:
enhanced_df = enhanced_df.drop(columns = ['DiagnosisDate', 
                                          'MetDiagnosisDate',
                                          'NephrectomyDate',
                                          'imported_StartDate'])

### Process Demographics.csv 

In [20]:
demographics_df = processor.process_demographics(file_path = '../data/Demographics.csv',
                                                 index_date_df = index_date_df,
                                                 index_date_column = 'StartDate')

2026-03-16 09:59:49,219 - INFO - Successfully read Demographics.csv file with shape: (10766, 6) and unique PatientIDs: 10766
2026-03-16 09:59:49,224 - WARNING - Found 1 ages outside valid range (18-120)
2026-03-16 09:59:49,227 - INFO - Successfully processed Demographics.csv file with final shape: (1668, 6) and unique PatientIDs: 1668


In [21]:
demographics_df = demographics_df.copy()

In [22]:
demographics_df = demographics_df.query('age >= 18 and age <= 120', engine = 'python')

In [23]:
# Impute missing with most common sex (male)
demographics_df['sex_male'] = np.where(demographics_df['Gender'] == 'F', 0, 1)

In [24]:
demographics_df = demographics_df.drop(columns = ['Gender'])

### Process Enhanced_MetRCCBiomarkers.csv

In [25]:
biomarkers_df = processor.process_biomarkers(file_path = '../data/Enhanced_MetRCCBiomarkers.csv',
                                             index_date_df = index_date_df, 
                                             index_date_column = 'StartDate',
                                             days_before = None, 
                                             days_after = 30)

2026-03-16 09:59:49,247 - INFO - Successfully read Enhanced_MetRCCBiomarkers.csv file with shape: (878, 17) and unique PatientIDs: 692
2026-03-16 09:59:49,251 - INFO - Successfully merged Enhanced_MetRCCBiomarkers.csv df with index_date_df resulting in shape: (119, 18) and unique PatientIDs: 94
2026-03-16 09:59:49,259 - INFO - Successfully processed Enhanced_MetRCCBiomarkers.csv file with final shape: (1668, 3) and unique PatientIDs: 1668


In [26]:
def map_pdl1(value):
    if pd.isna(value):  # leave missing as is
        return value
    elif value in ['0%', '< 1%']:
        return '0%'
    else:
        return '>=1%'

biomarkers_df['PDL1_binary'] = biomarkers_df['PDL1_percent_staining'].apply(map_pdl1)

In [27]:
biomarkers_df['PDL1_status'] = biomarkers_df['PDL1_status'].fillna('unknown')

In [28]:
biomarkers_df = biomarkers_df.drop(columns = ['PDL1_percent_staining'])

### Process ECOG.csv

In [29]:
ecog_df = processor.process_ecog(file_path = '../data/ECOG.csv', 
                                 index_date_df = index_date_df,
                                 index_date_column = 'StartDate',
                                 days_before = 90,
                                 days_after = 0,
                                 days_before_further = 180)

2026-03-16 09:59:49,315 - INFO - Successfully read ECOG.csv file with shape: (141710, 4) and unique PatientIDs: 7722
2026-03-16 09:59:49,336 - INFO - Successfully merged ECOG.csv df with index_date_df resulting in shape: (27078, 5) and unique PatientIDs: 1252
2026-03-16 09:59:49,360 - INFO - Successfully processed ECOG.csv file with final shape: (1668, 3) and unique PatientIDs: 1668


In [30]:
ecog_df['ecog_index'] = ecog_df['ecog_index'].astype('float64')
ecog_df['ecog_index_na'] = np.where(ecog_df['ecog_index'].isna(), 1, 0)

# impute 0 for missing ECOG since most common
ecog_df['ecog_index'] = ecog_df['ecog_index'].fillna(0)

In [31]:
ecog_df['ecog_newly_gte2'] = ecog_df['ecog_newly_gte2'].fillna(0)

### Process Vitals.csv

In [32]:
vitals_df = processor.process_vitals(file_path = '../data/Vitals.csv',
                                     index_date_df = index_date_df,
                                     index_date_column = 'StartDate',
                                     weight_days_before = 90,
                                     days_after = 0,
                                     vital_summary_lookback = 180, 
                                     abnormal_reading_threshold = 1)

2026-03-16 09:59:51,248 - INFO - Successfully read Vitals.csv file with shape: (2216004, 16) and unique PatientIDs: 10743
2026-03-16 09:59:52,034 - INFO - Successfully merged Vitals.csv df with index_date_df resulting in shape: (409935, 17) and unique PatientIDs: 1667
/Users/xavierorcutt/Dropbox/TrialRescue/myenv/lib/python3.13/site-packages/flatiron_cleaner/general.py:1085: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[None None]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  height_df.loc[mask_needs_imputation, 'TestResultCleaned'] = imputed_heights
2026-03-16 09:59:52,287 - INFO - Successfully processed Vitals.csv file with final shape: (1668, 8) and unique PatientIDs: 1668


### Process Lab.csv

In [33]:
labs_df = processor.process_labs(file_path = '../data/Lab.csv',
                                 index_date_df = index_date_df,
                                 index_date_column = 'StartDate',
                                 additional_loinc_mappings = {'neutrophil': ['26499-4', '751-8', '753-4']},
                                 days_before = 90,
                                 days_after = 0,
                                 summary_lookback = 180)

2026-03-16 10:00:01,166 - INFO - Successfully read Lab.csv file with shape: (7341289, 17) and unique PatientIDs: 10244
2026-03-16 10:00:02,602 - INFO - Successfully merged Lab.csv df with index_date_df resulting in shape: (1400302, 18) and unique PatientIDs: 1617
2026-03-16 10:00:05,428 - INFO - Successfully processed Lab.csv file with final shape: (1668, 81) and unique PatientIDs: 1668


### Process MedicationAdministration.csv

In [34]:
medications_df = processor.process_medications(file_path = '../data/MedicationAdministration.csv',
                                               index_date_df = index_date_df,
                                               index_date_column = 'StartDate',
                                               days_before = 90,
                                               days_after = 0)

2026-03-16 10:00:05,932 - INFO - Successfully read MedicationAdministration.csv file with shape: (407234, 11) and unique PatientIDs: 7413
2026-03-16 10:00:06,033 - INFO - Successfully merged MedicationAdministration.csv df with index_date_df resulting in shape: (71018, 12) and unique PatientIDs: 1348
2026-03-16 10:00:06,057 - INFO - Successfully processed MedicationAdministration.csv file with final shape: (1668, 9) and unique PatientIDs: 1668


### Process Diagnosis.csv

In [35]:
diagnosis_df = processor.process_diagnosis(file_path = '../data/Diagnosis.csv',
                                           index_date_df = index_date_df,
                                           index_date_column = 'StartDate',
                                           days_before = None,
                                           days_after = 0)

2026-03-16 10:00:06,278 - INFO - Successfully read Diagnosis.csv file with shape: (342031, 6) and unique PatientIDs: 10766
2026-03-16 10:00:06,317 - INFO - Successfully merged Diagnosis.csv df with index_date_df resulting in shape: (52403, 7) and unique PatientIDs: 1668
2026-03-16 10:00:06,489 - INFO - Successfully processed Diagnosis.csv file with final shape: (1668, 40) and unique PatientIDs: 1668


In [36]:
diagnosis_df['gi_met_combined'] = (
    diagnosis_df['adrenal_met'] | diagnosis_df['peritoneum_met'] | diagnosis_df['gi_met']
)

diagnosis_df = diagnosis_df.drop(columns = ['adrenal_met', 'peritoneum_met', 'gi_met'])

### Process Enhanced_Mortality_V2.csv

In [37]:
mortality_df = processor.process_mortality(file_path = '../data/Enhanced_Mortality_V2.csv',
                                           index_date_df = index_date_df, 
                                           index_date_column = 'StartDate',
                                           visit_path = '../data/Visit.csv', 
                                           telemedicine_path = '../data/Telemedicine.csv', 
                                           biomarkers_path = '../data/Enhanced_MetRCCBiomarkers.csv', 
                                           oral_path = '../data/Enhanced_MetRCC_Orals.csv',
                                           progression_path = '../data/Enhanced_MetRCCProgression.csv',
                                           drop_dates = False)

2026-03-16 10:00:06,501 - INFO - Successfully read Enhanced_Mortality_V2.csv file with shape: (6345, 2) and unique PatientIDs: 6345
2026-03-16 10:00:06,507 - INFO - Successfully merged Enhanced_Mortality_V2.csv df with index_date_df resulting in shape: (1668, 3) and unique PatientIDs: 1668
2026-03-16 10:00:06,780 - INFO - The following columns ['last_visit_date', 'last_biomarker_date', 'last_oral_date', 'last_progression_date'] are used to calculate the last EHR date
2026-03-16 10:00:06,783 - INFO - Successfully processed Enhanced_Mortality_V2.csv file with final shape: (1668, 6) and unique PatientIDs: 1668. There are 0 out of 1668 patients with missing duration values


In [38]:
mortality_df = mortality_df[['PatientID', 'event', 'duration']]

In [39]:
mortality_df = mortality_df.query('duration >= 0')

## Merge dataframes

In [40]:
testing_df = merge_dataframes(enhanced_df,
                              demographics_df,
                              biomarkers_df,
                              ecog_df,
                              vitals_df,
                              labs_df,
                              medications_df,
                              diagnosis_df,
                              mortality_df,
                              merge_type = 'inner')

2026-03-16 10:00:06,802 - INFO - Anticipated number of merges: 8
2026-03-16 10:00:06,802 - INFO - Anticipated number of columns in final dataframe presuming all columns are unique except for PatientID: 155
2026-03-16 10:00:06,803 - INFO - Dataset 1 shape: (1664, 11), unique PatientIDs: 1664
2026-03-16 10:00:06,804 - INFO - Dataset 2 shape: (1667, 6), unique PatientIDs: 1667
2026-03-16 10:00:06,804 - INFO - Dataset 3 shape: (1668, 3), unique PatientIDs: 1668
2026-03-16 10:00:06,805 - INFO - Dataset 4 shape: (1668, 4), unique PatientIDs: 1668
2026-03-16 10:00:06,805 - INFO - Dataset 5 shape: (1668, 8), unique PatientIDs: 1668
2026-03-16 10:00:06,805 - INFO - Dataset 6 shape: (1668, 81), unique PatientIDs: 1668
2026-03-16 10:00:06,806 - INFO - Dataset 7 shape: (1668, 9), unique PatientIDs: 1668
2026-03-16 10:00:06,806 - INFO - Dataset 8 shape: (1668, 38), unique PatientIDs: 1668
2026-03-16 10:00:06,807 - INFO - Dataset 9 shape: (1668, 3), unique PatientIDs: 1668
2026-03-16 10:00:06,811 - 

In [41]:
testing_df.shape

(1663, 155)

## Export dataframe

In [42]:
testing_df.to_csv('../outputs/1L_features_testing.csv', index = False)

In [43]:
# Save dtypes
testing_df.dtypes.apply(lambda x: x.name).to_csv('../outputs/1L_features_testing_dtypes.csv')